# Reusable template — binary logistic regression for security events

**Short name:** `LogReg_Cyber`  
Swap the path, target map, and feature list. Keep the order: *assume → fit → threshold → ROC → imbalance → simulate*.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, roc_auc_score,
)

# ---- edit these ----
DATA_PATH = "data/cyber_phishing.csv"
TARGET = "phishing"
POSITIVE_MAP = {"PHISH": 1, "BENIGN": 0}
FEATURES = ["url_length", "num_subdomains", "has_ip", "has_https", "digit_ratio"]
ID_COL = "event_id"
TEST_SIZE = 0.30
SEED = 0
THRESHOLD = 0.25          # SOC default in this lab — catch-rate first
# --------------------

df = pd.read_csv(DATA_PATH)
if df[TARGET].dtype == object or str(df[TARGET].dtype).startswith("string"):
    df[TARGET] = df[TARGET].map(POSITIVE_MAP).astype(int)

print(df[TARGET].value_counts())
if ID_COL and ID_COL in df.columns:
    print("unique ids == rows?", df[ID_COL].nunique() == df[ID_COL].count())
print("10-EPV max features", df[TARGET].value_counts().min() / 10)

Xtr, Xte, ytr, yte = train_test_split(
    df[FEATURES], df[TARGET], test_size=TEST_SIZE, random_state=SEED, stratify=df[TARGET]
)
model = LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000)
model.fit(Xtr, ytr)
proba = model.predict_proba(Xte)[:, 1]
pred = (proba >= THRESHOLD).astype(int)
print("coef", model.coef_, "intercept", model.intercept_)
print("acc", accuracy_score(yte, pred),
      "prec", precision_score(yte, pred, zero_division=0),
      "rec", recall_score(yte, pred, zero_division=0),
      "f1", f1_score(yte, pred, zero_division=0),
      "auc", roc_auc_score(yte, proba))
print(confusion_matrix(yte, pred))

fpr, tpr, _ = roc_curve(yte, proba)
plt.figure(figsize=(5, 5)); plt.plot(fpr, tpr); plt.plot([0, 1], [0, 1], "--")
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC")
plt.show()


## Optional extras

- Logit-linearity: `sns.regplot(x=col, y=TARGET, data=df, logistic=True)`
- Heatmap: `sns.heatmap(df[FEATURES].corr(), annot=True, center=0)`
- Balanced weights: `class_weight='balanced'`
- Threshold sweep: loop `t` and record FN/FP / tickets
- Simulation: resample rows, flip labels with `NOISE`, boxplot catch-rate
